# 05. #04 최적 하이퍼파라미터 + 멀티시드(5-seed) 앙상블

**배경**
- 현재 팀 최고 리더보드 기록은 #04(0.16927) — 19개 확정 파생변수 + `regularized` 계열 하이퍼파라미터 세밀 탐색 결과.
- 김효민님의 #07(LGBM+KNN+RF 이종 스태킹), #09(이종 멀티시드 + log1p 타겟 변환)는 오히려 단일 LightGBM보다 성능이 떨어짐 — 약한 모델을 섞거나 타겟을 변환하는 방향은 이 데이터에서 역효과.
- 반면 #10(**동일 모델, 랜덤 시드만 5개 다르게 해서 평균**)은 CV/리더보드 모두에서 안정적으로 성능을 개선한 유일한 앙상블 방식.

**이번 실험 방향**
새로운 이종 앙상블을 시도하는 대신, 실제 리더보드 1위 조합인 **#04의 하이퍼파라미터**에 **#10에서 검증된 멀티시드 평균 기법**을 그대로 적용합니다.
즉, 같은 LightGBM 구조/파라미터를 시드만 5개 바꿔 학습하고, OOF(out-of-fold) 예측과 최종 테스트 예측을 각각 평균냅니다.

비교 기준: #00 (0.2117) / #03 regularized (0.1855, LB 0.17688) / **#04 (0.1796, LB 0.16927, 현재 팀 최고)**


In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42  # 그라운드룰 1: KFold 분할 등 기준 시드는 항상 42로 고정 (재현성 보장)

# #10에서 검증된 방식: 모델 구조/파라미터는 고정하고 학습 시드만 5개로 다양화
# (그라운드룰 1과 상충되지 않음 - 아래 5개 시드 자체를 고정값으로 명시해 실험 전체는 100% 재현 가능)
SEEDS = [42, 52, 62, 72, 82]


## 1. Data Load

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('train:', train.shape, '/ test:', test.shape)


train: (3000, 18) / test: (3000, 17)


## 2. 결측치 및 중복행 처리 (0909 ver, 팀 확정본)

In [3]:
# 중복 행 제거 (ID 제외 기준) - train에만 적용, test는 제거하지 않음
train = train.drop_duplicates(
    subset=[col for col in train.columns if col not in ['ID']]
).reset_index(drop=True)

# 근로시간 결측치: 0 처리
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

# 범주형 결측치: 독립 범주 신설
for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

print('결측치 처리 후 남은 결측 개수 - train:', train.isnull().sum().sum(), '/ test:', test.isnull().sum().sum())
print('중복 제거 후 train shape:', train.shape)


결측치 처리 후 남은 결측 개수 - train: 0 / test: 0
중복 제거 후 train shape: (2994, 18)


## 3. 파생변수 생성 (19개 확정본)

`파생변수_생성_0909ver_19개.md`의 최종 확정 함수 그대로 사용 (`activity_sleep_mismatch` 포함 19개 전부).

In [4]:
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    # 1. 과로 및 생활 리듬
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    # 2. 질환 및 유전력
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    # 3. 심혈관 및 신체
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    # 4. 대사 및 노화
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data


train = add_features(train)
test = add_features(test)

print('파생변수 추가 후 train shape:', train.shape, '(19개 확정)')


파생변수 추가 후 train shape: (2994, 37) (19개 확정)


## 4. 인코딩 (#00과 동일: Ordinal + LabelEncoder 혼합)

In [5]:
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}

train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])

    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen:
        le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

print('x_train:', x_train.shape, '/ x_test:', x_test.shape)


x_train: (2994, 35) / x_test: (3000, 35)


## 5. #04 최적 하이퍼파라미터로 멀티시드 Out-of-Fold CV

#04에서 20개 조합 랜덤 탐색으로 찾은 최적값을 그대로 고정하고, 학습 시드만 `SEEDS`의 5개로 바꿔가며 같은 KFold 분할(`random_state=RANDOM_STATE`)에 대해 OOF 예측을 만듭니다.
폴드별로 5개 시드 예측을 평균낸 뒤 MAE를 계산하므로, 이 점수는 "실제 제출 시 5개 시드 평균 예측이 낼 성능"을 그대로 반영하는 정직한 CV입니다.

In [6]:
# #04에서 확정된 최적 하이퍼파라미터 (random_state는 아래에서 시드별로 덮어씀)
best_params_04 = dict(
    n_estimators=1200,
    learning_rate=0.08,
    reg_alpha=0.15,
    reg_lambda=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_splits = list(kf.split(x_train))

oof_preds_per_seed = np.zeros((len(x_train), len(SEEDS)))

for si, seed in enumerate(SEEDS):
    for tr_idx, val_idx in fold_splits:
        X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = LGBMRegressor(**best_params_04, random_state=seed, verbose=-1)
        model.fit(X_tr, y_tr)
        oof_preds_per_seed[val_idx, si] = model.predict(X_val)

    seed_mae = mean_absolute_error(y_train, oof_preds_per_seed[:, si])
    print(f'seed={seed} 단일 시드 OOF MAE = {seed_mae:.4f}')


seed=42 단일 시드 OOF MAE = 0.1815
seed=52 단일 시드 OOF MAE = 0.1825
seed=62 단일 시드 OOF MAE = 0.1809
seed=72 단일 시드 OOF MAE = 0.1806
seed=82 단일 시드 OOF MAE = 0.1804


In [7]:
avg_oof_pred = oof_preds_per_seed.mean(axis=1)
multiseed_cv_mae = mean_absolute_error(y_train, avg_oof_pred)

print()
print(f'=== 5-seed 평균 OOF MAE = {multiseed_cv_mae:.4f} ===')
print()
print('비교: #00 0.2117 / #03 0.1855 / #04(단일 시드) 0.1796')
print('#04 대비 개선되었는지 확인 후, 개선된 경우에만 그라운드룰 3(로컬 CV 우선주의)에 따라 리더보드에 제출하세요.')



=== 5-seed 평균 OOF MAE = 0.1794 ===

비교: #00 0.2117 / #03 0.1855 / #04(단일 시드) 0.1796
#04 대비 개선되었는지 확인 후, 개선된 경우에만 그라운드룰 3(로컬 CV 우선주의)에 따라 리더보드에 제출하세요.


## 6. 최종 학습 (5-seed) + 테스트 예측 평균 + 제출 파일 저장

In [8]:
import os

sample_submission = pd.read_csv('../data/sample_submission.csv')

test_preds_per_seed = np.zeros((len(x_test), len(SEEDS)))

for si, seed in enumerate(SEEDS):
    final_model = LGBMRegressor(**best_params_04, random_state=seed, verbose=-1)
    final_model.fit(x_train, y_train)
    test_preds_per_seed[:, si] = final_model.predict(x_test)
    print(f'seed={seed} 전체 데이터 학습 및 예측 완료')

final_pred = test_preds_per_seed.mean(axis=1)

os.makedirs('../submissions', exist_ok=True)
sample_submission['stress_score'] = final_pred
sample_submission.to_csv('../submissions/submit_05_multiseed_ensemble.csv', index=False)

print()
print('제출 파일 저장 완료: ../submissions/submit_05_multiseed_ensemble.csv')
sample_submission.head()


seed=42 전체 데이터 학습 및 예측 완료
seed=52 전체 데이터 학습 및 예측 완료
seed=62 전체 데이터 학습 및 예측 완료
seed=72 전체 데이터 학습 및 예측 완료
seed=82 전체 데이터 학습 및 예측 완료

제출 파일 저장 완료: ../submissions/submit_05_multiseed_ensemble.csv


,ID,stress_score
0,TEST_0000,0.604379
1,TEST_0001,0.976296
2,TEST_0002,0.234789
3,TEST_0003,0.492449
4,TEST_0004,0.571211


## 실험 로그 기록 안내

그라운드룰 2(실험 로그 의무화)에 따라, 이 노트북 실행 후 아래 정보를 노션 실험 표에 기록해 주세요.

- 모델: LGBMRegressor (5-seed 평균, `SEEDS = [42, 52, 62, 72, 82]`)
- 전처리/파생변수: 0909ver 19개 확정본 (동일)
- 하이퍼파라미터: #04 최적값 그대로 (n_estimators=1200, learning_rate=0.08, reg_alpha=0.15, reg_lambda=0.1, subsample=0.8, colsample_bytree=0.8)
- 5-Fold CV MAE: 위 셀 출력값(`multiseed_cv_mae`) 기록
- 데이콘 리더보드: 제출 후 점수 기록 (#04의 0.16927보다 개선되는지 확인)
